# Barnstable and Long-Run Risk

**FINM 36700 · Homework 3 · Group 22**  
Lucas Albín Lazcano, Avi Rastogi, Mengting Wang, and Yihan Kong

*Acknowledgements: ChatGPT 5.6 was used to help explain the concepts, organize the submission, and assist with coding the solutions.*

# 2. Estimating Underperformance

### Data

Use the returns on the S&P 500 ($r^m$) and 1-month T-bills ($r^f$) provided in `data/barnstable_analysis_data.xlsx` in the course Box folder.

- Monthly data covers January 1926 through December 2025.
- Stock and T-bill observations are aligned by calendar month. Both series cover all 1,200 months.

Barnstable's estimates of mean and volatility are based on the subsample of 1965 to 1999.

- We consider this subsample, as well as 2000–2025, as well as the full sample of 1926–2025.

### Notation

- $r$ = level return rates
- $R$ = cumulative return factor
- $\mathtt{r}$ = log return rates

$$
R\equiv 1+r
$$

$$
\mathtt{r}\equiv\ln(1+r)=\ln(R)
$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

FILE_PATH = "barnstable_analysis_data.xlsx"

descriptions = pd.read_excel(FILE_PATH, sheet_name="descriptions")
returns = pd.read_excel(FILE_PATH, sheet_name="data")

descriptions

In [ ]:
returns.head()

## 1. Summary Statistics

Report the following (annualized) statistics.

| Return series | 1965–1999 mean | 1965–1999 vol | 2000–2025 mean | 2000–2025 vol | 1926–2025 mean | 1926–2025 vol |
|:---|---:|---:|---:|---:|---:|---:|
| Level market return, $r^m$ | | | | | | |
| Level market excess return, $\widetilde r^m$ | | | | | | |
| Level risk-free return, $r^f$ | | | | | | |
| Log market return, $\mathtt r^m$ | | | | | | |
| Log market excess return, $\widetilde{\mathtt r}^m$ | | | | | | |
| Log risk-free return, $\mathtt r_f$ | | | | | | |

- Comment on how the full-sample return stats compare to the sub-sample stats.
- Comment on how the level stats compare to the log stats.

In [ ]:
returns = returns.set_index("date")

rm = returns["SPX"]
rf = returns["TB1M"]

series = pd.DataFrame({
    "r^m": rm,
    "r~^m": rm - rf,
    "r^f": rf,
    "log r^m": np.log1p(rm),
    "log r~^m": np.log1p(rm) - np.log1p(rf),
    "log r^f": np.log1p(rf),
})

In [ ]:
samples = {
    "1965-1999": ("1965", "1999"),
    "2000-2025": ("2000", "2025"),
    "1926-2025": ("1926", "2025"),
}

summary = pd.concat({
    name: pd.DataFrame({
        "mean": series.loc[start:end].mean() * 12,
        "vol": series.loc[start:end].std() * np.sqrt(12),
    })
    for name, (start, end) in samples.items()
}, axis=1)

summary.round(4)

_Answer:_ The full-sample market mean is 11.61%, which lies between the 1965–1999 mean of 12.94% and the 2000–2025 mean of 9.08%. The full sample has substantially higher market volatility: 18.60%, compared with 14.94% in 1965–1999 and 15.14% in 2000–2025. This indicates that the 1926–2025 sample contains more extreme market movements than either of the two shorter subsamples.

The lower total market return after 1999 does not imply that the market earned a lower excess return. The mean level excess return increases slightly from 6.79% in 1965–1999 to 7.26% in 2000–2025. The main difference is that the average risk-free return falls sharply from 6.15% to 1.83%.

Mean log returns are lower than mean level returns in every sample. For example, the full-sample market mean is 11.61% in levels and 9.85% in logs. This is consistent with $\ln(1+r)\approx r-\frac{1}{2}r^2$, so the difference becomes larger when volatility is higher. Level and log volatilities remain close because monthly returns are generally small.

## 2. Probability of Underperformance

Recall the following:

- If $x\sim\mathcal N(\mu_x,\sigma_x^2)$, then

$$
\Pr[x<\ell]=\Phi_{\mathcal N}(L),
$$

where $L=\frac{\ell-\mu_x}{\sigma_x}$ and $\Phi_{\mathcal N}$ denotes the standard normal cdf.

- Cumulative log returns are the sum of the single-period log returns:

$$
\mathtt r^m_{t,t+h}\equiv\sum_{i=1}^{h}\mathtt r^m_{t+i}.
$$

- The $h$-period average ending at time $t+h$ is

$$
\overline{\mathtt r}^{,m}_{t,t+h}
=\frac{1}{h}\sum_{i=1}^{h}\mathtt r^m_{t+i}.
$$

Calculate the probability that the cumulative market return will fall short of the cumulative risk-free return:

$$
\Pr\left[R^m_{t,t+h}<R^f_{t,t+h}\right].
$$

To analyze this analytically, convert the probability statement above to a probability statement about mean log returns.

_Answer:_ Since the logarithm is increasing,

$$
R^m_{t,t+h}<R^f_{t,t+h}
\iff
\sum_{i=1}^{h}\left(\mathtt r^m_{t+i}-\mathtt r^f_{t+i}\right)<0
\iff
\overline{\widetilde{\mathtt r}}^{,m}_{t,t+h}<0.
$$

Thus, the market underperforms the risk-free asset when its average log excess return over the investment horizon is negative. If annual log excess returns are i.i.d. normal with mean $\mu$ and volatility $\sigma$, then

$$
\Pr\left[\overline{\widetilde{\mathtt r}}^{,m}_{t,t+h}<0\right]
=\Phi_{\mathcal N}\left(-\sqrt h\frac{\mu}{\sigma}\right).
$$

The ratio $\mu/\sigma$ is the Sharpe ratio of log excess returns, so a higher Sharpe ratio implies a lower probability of long-horizon underperformance.

### 2.1

Calculate the probability using the subsample 1965–1999.

In [ ]:
subsample = series.loc["1965":"1999", "log r~^m"]

mu = subsample.mean() * 12
sigma = subsample.std() * np.sqrt(12)
sharpe_ratio = mu / sigma

print(f"Annualized mean: {mu:.4%}")
print(f"Annualized volatility: {sigma:.4%}")
print(f"Sharpe ratio: {sharpe_ratio:.4f}")

_Answer:_ The estimated annualized mean and volatility of log excess returns over 1965–1999 are 5.6301% and 15.0286%. Their ratio gives a log excess Sharpe ratio of 0.3746. Therefore,

$$
p(h)=\Phi_{\mathcal N}\left(-0.3746\sqrt h\right).
$$

This equation gives the estimated probability of market underperformance for any horizon $h$, measured in years, under the normal and i.i.d. return assumptions.

### 2.2

Report the precise probability for $h=15$ and $h=30$ years.

In [ ]:
def p_under(h, sr=sharpe_ratio):
    return norm.cdf(-np.sqrt(h) * sr)


print(f"15 years: {p_under(15):.4%}")
print(f"30 years: {p_under(30):.4%}")

_Answer:_ The probability of underperformance is 7.3402% over 15 years and 2.0089% over 30 years. The estimated probability declines as the horizon increases because positive expected excess returns accumulate in proportion to $h$, while cumulative volatility increases only with $\sqrt h$. Under the model, the mean therefore becomes large relative to the standard deviation at longer horizons.

### 2.3

Plot the probability as a function of the investment horizon, $h$, for $0<h\leq30$ years.

**Hint:** The probability can be expressed as

$$
p(h)=\Phi_{\mathcal N}\left(-\sqrt h\,\mathrm{SR}\right),
$$

where $\mathrm{SR}$ denotes the sample Sharpe ratio of log market returns.

In [ ]:
h = np.linspace(0.01, 30, 500)

plt.figure(figsize=(8, 5))
plt.plot(h, p_under(h))
plt.xlabel("Investment horizon (years)")
plt.ylabel("Probability of underperformance")
plt.title("Probability of Market Underperformance")
plt.grid(alpha=0.3)
plt.show()

_Answer:_ The graph shows that the estimated probability of underperformance falls as the investment horizon increases. The decline is relatively steep at shorter horizons and becomes flatter as the horizon approaches 30 years. Using the 1965–1999 estimates, the probability falls from close to 50% at a very short horizon to approximately 2% at 30 years. This result depends on the assumptions that the mean and volatility remain constant and that returns are independent over time.

## 3. Full Sample Analysis

Use the sample 1965–2025 to reconsider the 30-year probability. As of the end of 2025, calculate the probability of the stock return underperforming the risk-free rate over the next 30 years. That is, $R^m_{t,t+h}$ underperforming $R^f_{t,t+h}$ for $0<h\leq30$.

In [ ]:
updated_sample = series.loc["1965":"2025", "log r~^m"]

mu_updated = updated_sample.mean() * 12
sigma_updated = updated_sample.std() * np.sqrt(12)
sr_updated = mu_updated / sigma_updated
p_30_updated = p_under(30, sr=sr_updated)

print(f"Annualized mean: {mu_updated:.4%}")
print(f"Annualized volatility: {sigma_updated:.4%}")
print(f"Sharpe ratio: {sr_updated:.4f}")
print(f"30-year probability: {p_30_updated:.4%}")

_Answer:_ Using the 1965–2025 sample, the annualized mean log excess return is 5.8215% and its annualized volatility is 15.1141%, producing a Sharpe ratio of 0.3852. The estimated probability of market underperformance over the next 30 years is therefore 1.7444%.

This is slightly below the 2.0089% probability obtained from the 1965–1999 sample. The additional 2000–2025 observations raise the estimated log excess Sharpe ratio slightly, which reduces the model-implied probability of underperformance.

## 4. In-Sample Estimate of Out-of-Sample Likelihood

Let's consider how things turned out relative to Barnstable's 1999 expectations.

What was the probability (based on the 1999 estimate of $\mu$) that the $h$-year market return, $R^m_{t,t+h}$, would be smaller than that realized in 2000–2025?

**Hint:** You can calculate this as

$$
p=\Phi_{\mathcal N}\left(
\sqrt h\,
\frac{
\overline{\mathtt r}_{\text{out-of-sample}}
-\overline{\mathtt r}_{\text{in-sample}}
}{\sigma_{\text{in-sample}}}
\right),
$$

where “in-sample” denotes 1965–1999 and “out-of-sample” denotes 2000–2025 ($h=26$ years).

In [ ]:
in_sample = series.loc["1965":"1999", "log r^m"]
out_of_sample = series.loc["2000":"2025", "log r^m"]

mu_in = in_sample.mean() * 12
sigma_in = in_sample.std() * np.sqrt(12)
mu_out = out_of_sample.mean() * 12
h = 26

z = np.sqrt(h) * (mu_out - mu_in) / sigma_in
probability = norm.cdf(z)

print(f"In-sample mean: {mu_in:.4%}")
print(f"Out-of-sample mean: {mu_out:.4%}")
print(f"In-sample volatility: {sigma_in:.4%}")
print(f"z-score: {z:.4f}")
print(f"Probability: {probability:.4%}")

_Answer:_ The annualized mean market log return is 11.7600% in the 1965–1999 in-sample period and 7.9031% in the 2000–2025 out-of-sample period. Using the in-sample volatility of 14.9568% and a 26-year horizon gives a z-score of -1.3149 and a probability of 9.4276%.

Thus, based on Barnstable's 1999 estimates, a 26-year average market return as low as the one realized during 2000–2025 was a relatively weak lower-tail outcome. However, this calculation concerns the total market return. Part of the decline in total returns reflects the much lower risk-free rate after 1999, rather than an equally large decline in the market's excess return.